In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\amit6\OneDrive\Desktop\Intelligent-Retail-Customer-Analytics\Data\Processed_data\retail_feature_engineered.csv")
df.head()

C:\Users\amit6\AppData\Local\Temp\ipykernel_16764\550069564.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\amit6\OneDrive\Desktop\Intelligent-Retail-Customer-Analytics\Data\Processed_data\retail_feature_engineered.csv")


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalAmount,Year,...,Hour,IsWeekend,InvoiceTotal,BasketSize,PurchaseFrequency,AverageOrderValue,CustomerRevenue,ProductPurchaseCount,ProductRevenue,CustomerType
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009,...,7,False,505.3,8,8.0,350.900238,2433.28,522,21201.13,Regular
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,...,7,False,505.3,8,8.0,350.900238,2433.28,266,13787.93,Regular
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,...,7,False,505.3,8,8.0,350.900238,2433.28,350,18048.72,Regular
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009,...,7,False,505.3,8,8.0,350.900238,2433.28,560,18650.87,Regular
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009,...,7,False,505.3,8,8.0,350.900238,2433.28,2478,49300.57,Regular


In [3]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [4]:
snapshot_date = df["InvoiceDate"].max()

print(snapshot_date)

2011-12-09 12:50:00


In [5]:
customer_features = df.groupby("Customer ID").agg({

    "Invoice": "nunique",

    "Quantity": "sum",

    "TotalAmount": "sum",

    "InvoiceDate": ["min", "max"],

    "Price": "mean",

    "StockCode": "nunique",

    "Country": "first"

}).reset_index()

In [6]:
customer_features.columns = [

    "CustomerID",

    "Frequency",

    "TotalQuantity",

    "Revenue",

    "FirstPurchase",

    "LastPurchase",

    "AverageUnitPrice",

    "UniqueProducts",

    "Country"

]

In [7]:
customer_features["CustomerTenure"] = (
    customer_features["LastPurchase"] -
    customer_features["FirstPurchase"]
).dt.days

In [8]:
customer_features.head()

,CustomerID,Frequency,TotalQuantity,Revenue,FirstPurchase,LastPurchase,AverageUnitPrice,UniqueProducts,Country,CustomerTenure
0,12346.0,12,74285,77556.46,2009-12-14 08:34:00,2011-01-18 10:01:00,6.100000,27,United Kingdom,400
1,12347.0,8,2967,4921.53,2010-10-31 14:20:00,2011-12-07 15:52:00,2.498063,126,Iceland,402
2,12348.0,5,2714,2019.40,2010-09-27 14:59:00,2011-09-25 13:13:00,3.786275,25,Finland,362
3,12349.0,4,1624,4428.69,2010-04-29 13:20:00,2011-11-21 09:51:00,8.459657,138,Italy,570
4,12350.0,1,197,334.40,2011-02-02 16:01:00,2011-02-02 16:01:00,3.841176,17,Norway,0


In [9]:
customer_features["Recency"] = (
    snapshot_date -
    customer_features["LastPurchase"]
).dt.days

In [10]:
customer_features["AverageOrderValue"] = (
    customer_features["Revenue"] /
    customer_features["Frequency"]
)

In [11]:
customer_features["AverageQuantityPerOrder"] = (
    customer_features["TotalQuantity"] /
    customer_features["Frequency"]
)

In [12]:
weekend_ratio = (
    df.groupby("Customer ID")["IsWeekend"]
    .mean()
    .reset_index())

In [13]:
weekend_ratio.columns = [
    "CustomerID",
    "WeekendPurchaseRatio"
]

In [14]:
customer_features = customer_features.merge(
    weekend_ratio,
    on="CustomerID",
    how="left"
)

In [15]:
returns = (

    df.assign(Return=df["Quantity"] < 0).groupby("Customer ID")["Return"].mean().reset_index()

)

In [16]:
returns.columns = [

    "CustomerID",

    "ReturnRate"

]

In [17]:
customer_features = customer_features.merge(

    returns,

    on="CustomerID",

    how="left"

)

In [18]:
purchase_gap = (

    df[["Customer ID","InvoiceDate"]]

    .drop_duplicates()

    .sort_values(["Customer ID","InvoiceDate"])

)

In [19]:
purchase_gap["Gap"] = (

purchase_gap

.groupby("Customer ID")["InvoiceDate"]

.diff()

.dt.days

)

In [20]:
gap = (

purchase_gap

.groupby("Customer ID")["Gap"]

.mean()

.reset_index()

)

In [21]:
gap.columns = [

    "CustomerID",

    "AverageDaysBetweenOrders"

]

In [22]:
customer_features = customer_features.merge(

gap,

on="CustomerID",

how="left"

)

In [23]:
customer_features.isnull().sum()

CustomerID                     0
Frequency                      0
TotalQuantity                  0
Revenue                        0
FirstPurchase                  0
LastPurchase                   0
AverageUnitPrice               0
UniqueProducts                 0
Country                        0
CustomerTenure                 0
Recency                        0
AverageOrderValue              0
AverageQuantityPerOrder        0
WeekendPurchaseRatio           0
ReturnRate                     0
AverageDaysBetweenOrders    1622
dtype: int64

In [24]:
customer_features["AverageDaysBetweenOrders"] = (

customer_features["AverageDaysBetweenOrders"]

.fillna(0)

)

In [28]:
customer_type = (
    df.groupby("Customer ID")["CustomerType"]
      .first()
)

customer_features["CustomerType"] = (
    customer_features["CustomerID"]
    .map(customer_type)
)

In [30]:
customer_features.to_csv(

"customer_feature_store.csv",

index=False

)

In [29]:
customer_features.head()

,CustomerID,Frequency,TotalQuantity,Revenue,FirstPurchase,LastPurchase,AverageUnitPrice,UniqueProducts,Country,CustomerTenure,Recency,AverageOrderValue,AverageQuantityPerOrder,WeekendPurchaseRatio,ReturnRate,AverageDaysBetweenOrders,CustomerType
0,12346.0,12,74285,77556.46,2009-12-14 08:34:00,2011-01-18 10:01:00,6.100000,27,United Kingdom,400,325,6463.038333,6190.416667,0.000000,0.0,35.909091,Loyal
1,12347.0,8,2967,4921.53,2010-10-31 14:20:00,2011-12-07 15:52:00,2.498063,126,Iceland,402,1,615.191250,370.875000,0.180180,0.0,57.000000,Regular
2,12348.0,5,2714,2019.40,2010-09-27 14:59:00,2011-09-25 13:13:00,3.786275,25,Finland,362,74,403.880000,542.800000,0.058824,0.0,90.500000,Occasional
3,12349.0,4,1624,4428.69,2010-04-29 13:20:00,2011-11-21 09:51:00,8.459657,138,Italy,570,18,1107.172500,406.000000,0.000000,0.0,189.666667,Occasional
4,12350.0,1,197,334.40,2011-02-02 16:01:00,2011-02-02 16:01:00,3.841176,17,Norway,0,309,334.400000,197.000000,0.000000,0.0,0.000000,New


In [27]:
customer_features.columns

Index(['CustomerID', 'Frequency', 'TotalQuantity', 'Revenue', 'FirstPurchase',
       'LastPurchase', 'AverageUnitPrice', 'UniqueProducts', 'Country',
       'CustomerTenure', 'Recency', 'AverageOrderValue',
       'AverageQuantityPerOrder', 'WeekendPurchaseRatio', 'ReturnRate',
       'AverageDaysBetweenOrders'],
      dtype='object')